In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.dropdown(
    "file_name",
    "customers",
    ["customers", "orders", "products", "regions"]
)

dbutils.widgets.text("storage_account_name", "datalakezakariae2026")
dbutils.widgets.text("source_container", "source")
dbutils.widgets.text("target_container", "bronze")

In [0]:
fileName = dbutils.widgets.get("file_name").strip()
storageAccountName = dbutils.widgets.get("storage_account_name").strip()
sourceContainerName = dbutils.widgets.get("source_container").strip()
targetContainerName = dbutils.widgets.get("target_container").strip()

sourcePath = (
    f"abfss://{sourceContainerName}@{storageAccountName}.dfs.core.windows.net/{fileName}"
)

bronzePath = (
    f"abfss://{targetContainerName}@{storageAccountName}.dfs.core.windows.net/{fileName}"
)

{
    "source_path": sourcePath,
    "bronze_path": bronzePath
}

{'source_path': 'abfss://source@datalakezakariae2026.dfs.core.windows.net/products',
 'bronze_path': 'abfss://bronze@datalakezakariae2026.dfs.core.windows.net/products'}

In [0]:
sourceDf = (
    spark.read
    .format("parquet")
    .option("recursiveFileLookup", "true")
    .load(sourcePath)
)

sourceCount = sourceDf.count()

sourceCount

500

In [0]:
def listParquetFiles(folderPath):
    parquetFilePaths = []

    for fileInfo in dbutils.fs.ls(folderPath):
        folderOrFileName = fileInfo.name.rstrip("/")

        if fileInfo.name.endswith("/"):
            if not folderOrFileName.startswith("_") and not folderOrFileName.startswith("."):
                parquetFilePaths.extend(
                    listParquetFiles(fileInfo.path)
                )
        elif fileInfo.path.endswith(".parquet"):
            parquetFilePaths.append(fileInfo.path)

    return parquetFilePaths


bronzeParquetFiles = listParquetFiles(bronzePath)

if not bronzeParquetFiles:
    raise ValueError(
        f"No parquet files found in bronze path: {bronzePath}"
    )

bronzeDf = (
    spark.read
    .format("parquet")
    .load(bronzeParquetFiles)
)

bronzeCount = bronzeDf.count()

bronzeCount

10

In [0]:
comparisonResult = {
    "file_name": fileName,
    "source_count": sourceCount,
    "bronze_count": bronzeCount,
    "is_equal": sourceCount == bronzeCount,
    "difference": sourceCount - bronzeCount
}

comparisonResult

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8570130615001415>, line 2
      1 comparisonResult = {
----> 2     "file_name": fileName,
      3     "source_count": sourceCount,
      4     "bronze_count": bronzeCount,
      5     "is_equal": sourceCount == bronzeCount,
      6     "difference": sourceCount - bronzeCount
      7 }
      9 comparisonResult

NameError: name 'fileName' is not defined

In [0]:
storageAccountName = "datalakezakariae2026"
fileName = "products"

sourcePath = (
    f"abfss://source@{storageAccountName}.dfs.core.windows.net/{fileName}"
)

bronzePath = (
    f"abfss://bronze@{storageAccountName}.dfs.core.windows.net/{fileName}"
)

sourceDf = (
    spark.read
    .format("parquet")
    .option("recursiveFileLookup", "true")
    .load(sourcePath)
)

bronzeDf = (
    spark.read
    .format("delta")
    .load(bronzePath)
)

comparisonResult = {
    "file_name": fileName,
    "source_count": sourceDf.count(),
    "bronze_count": bronzeDf.count()
}

comparisonResult

{'file_name': 'products', 'source_count': 500, 'bronze_count': 500}